In [2]:
import os
import sys
sys.path.append(os.path.abspath('../scripts'))
fig_path      = '../figures/'
data_path     = '../data/'

In [3]:
import netCDF4 as nc
import numpy as np
import xarray as xr
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd 
import seaborn as sns
import cmocean as cmo
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from proj_utils import *
from mapping_utils import *

## (1) Open original SLP file, to get LSM

In [6]:
ds_slp = xr.open_dataset(data_path + 'north_atlantic_slp.nc')

ds_slp = ds_slp.sel(expver = slice(0,1)).squeeze()
ds_slp = ds_slp.drop_vars('expver')
ds_slp = ds_slp.sel(time = slice('1993-01-01','2022-12-01'))
ds_slp = ds_slp.sortby(ds_slp.latitude)

ds_slp = seasonal_detrend(ds_slp)

In [8]:
lsm_time_mean = ds_slp.lsm.mean(dim = 'time')

## (2) Open new SLP file

In [22]:
ds = xr.open_dataset(data_path + 'era5_north_atlantic_slp_monthly_1940_2023.nc')

## (3) Process to my usual conventions

In [23]:
ds = ds.drop_vars(('expver', 'number'))                                  # Drop unneeded vars
ds = ds.rename({'valid_time': 'time'})                                   # Rename time dimension
ds = ds.sortby(ds.latitude)                                              # Sort latitude ascending
ds = ds.sel(latitude = slice(ds_slp.latitude[0],ds_slp.latitude[-1]), 
            longitude = slice(ds_slp.longitude[0],ds_slp.longitude[-1])) # Trim to bounds of original ds
ds['lsm'] = lsm_time_mean                                                # Add time mean land sea mask

In [30]:
ds.to_netcdf(data_path + 'north_atlantic_slp_1940_2023.nc')